# Lecture live demonstrations

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

Every piece of code demonstrated live in lectures, in one notebook, sectioned by week. Slides carry the ideas; this notebook carries the code.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of the craft this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Colab already has pandas, numpy, matplotlib, statsmodels, scipy and sklearn.
# This notebook additionally needs: linearmodels, duckdb.
%pip install -q linearmodels duckdb

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

# W01 — Financial Data in the Age of AI

### W01 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**The unit in fifteen lines**

In [ ]:
import matplotlib.pyplot as plt

px = pd.read_parquet(DATA / "stocks_daily.parquet")   # 7 tickers, daily, 2010 onwards
spy = px.loc[px["ticker"] == "SPY"].sort_values("date").copy()

spy["ret"] = spy["adj_close"].pct_change()            # simple daily return
spy["growth"] = (1 + spy["ret"].fillna(0)).cumprod()  # compound: growth of £1

final = spy["growth"].iloc[-1]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(spy["date"], spy["growth"], color="#0F6BBA", lw=1.5)
ax.set_title(f"£1 in SPY since 2010 finishes at £{final:.2f}")
ax.set_ylabel("Value of £1")
plt.show()

**Villain 1: survivorship bias**

In [ ]:
import matplotlib.pyplot as plt

winners = ["AAPL", "MSFT", "NVDA"]
prices = (px[px["ticker"].isin(winners + ["SPY"])]
          .pivot(index="date", columns="ticker", values="adj_close")
          .sort_index())
rets = prices.pct_change().dropna()

port = (1 + rets[winners].mean(axis=1)).cumprod()     # EW, rebalanced daily
spy_g = (1 + rets["SPY"]).cumprod()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(port.index, port, color="#0F766E", lw=1.5, label="Today's giants (EW)")
ax.plot(spy_g.index, spy_g, color="#0F6BBA", lw=1.5, label="SPY")
ax.set_yscale("log")
ax.set_ylabel("Value of £1 (log scale)")
ax.legend()
plt.show()

**Villain 2: look-ahead bias**

In [ ]:
import matplotlib.pyplot as plt

r = rets["SPY"]
cheat = (1 + r.where(r > 0, 0.0)).cumprod()           # uses today's close — impossible
honest = (1 + r.where(r.shift(1) > 0, 0.0)).cumprod() # uses yesterday's — feasible
bh = (1 + r).cumprod()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cheat.index, cheat, color="#B91C1C", lw=1.5, label="Cheat: today's return")
ax.plot(honest.index, honest, color="#0F766E", lw=1.5, label="Honest: lagged signal")
ax.plot(bh.index, bh, color="#0F6BBA", lw=1.5, label="Buy and hold")
ax.set_yscale("log")
ax.set_ylabel("Value of £1 (log scale)")
ax.set_title(f"One missing shift(1) turns £1 into £{cheat.iloc[-1]:,.0f}")
ax.legend()
plt.show()

# W02 — Python Essentials for Financial Data

### W02 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**Python is a calculator with a memory**

In [ ]:
price_of_apple = 171.30
shares_held = 120
price_of_apple * shares_held

**Every value has a type**

In [ ]:
type(171.30), type(120), type("AAPL"), type(True)

**Every value has a type**

In [ ]:
"100" == 100

**Lists: ordered collections**

In [ ]:
closes = [100.0, 101.5, 99.8, 102.3, 103.1]
len(closes)

**Indexing: Python counts from 0**

In [ ]:
closes[0]     # the FIRST element

**Indexing: Python counts from 0**

In [ ]:
closes[-1]    # the last element (negative = count from the end)

**Indexing: Python counts from 0**

In [ ]:
closes[1:3]   # positions 1 and 2 — two elements, not three

**Dictionaries: labelled values**

In [ ]:
position = {"ticker": "AAPL", "shares": 120, "avg_cost": 171.30}
position["shares"]

**Functions: naming a recipe**

In [ ]:
def position_value(shares, price):
    """Market value of a holding."""
    return shares * price

position_value(120, 171.30)

**numpy: arithmetic on whole columns at once**

In [ ]:
prices = np.array([100.0, 101.5, 99.8, 102.3])
prices * 2

**A Series is values *plus* an index**

In [ ]:
rets = pd.Series([0.012, -0.004, 0.008],
                 index=pd.to_datetime(["2024-01-31", "2024-02-29", "2024-03-29"]))
rets

**The index drives alignment**

In [ ]:
fund_a = pd.Series({"AAPL": 0.02, "MSFT": 0.01})
fund_b = pd.Series({"MSFT": 0.03, "NVDA": 0.05})
fund_a + fund_b

**A DataFrame is a dict of Series sharing one index**

In [ ]:
df = pd.DataFrame({"open": [188.2, 190.1], "close": [189.7, 189.9]},
                  index=pd.to_datetime(["2024-01-02", "2024-01-03"]))
df["close"]           # one column = one Series

**Selecting rows: `.loc` vs `.iloc`**

In [ ]:
df.iloc[0:1]                          # by POSITION: half-open, row 0 only

**Selecting rows: `.loc` vs `.iloc`**

In [ ]:
df.loc["2024-01-02":"2024-01-03"]     # by LABEL: inclusive of BOTH ends

**Reading real data: parquet**

In [ ]:
stocks = pd.read_parquet(DATA / "stocks_daily.parquet")
stocks.head()

**Always check the dtypes**

In [ ]:
stocks.dtypes

**Poll reveal**

In [ ]:
s = pd.Series(["9", "10", "100"])
s.max()

**The object-dtype trap, loud version**

In [ ]:
raw = pd.Series([101.2, 99.8, "100", 102.5])
raw.dtype

**The object-dtype trap, loud version**

In [ ]:
raw.mean()

**The fix: convert deliberately**

In [ ]:
clean = pd.to_numeric(raw)     # errors="raise" by default: bad values stop you
clean.mean()

**Dates are not strings either**

In [ ]:
pd.to_datetime("31/01/2024", dayfirst=True)

**From daily to month-end**

In [ ]:
spy = stocks.loc[stocks["ticker"] == "SPY", ["date", "adj_close"]]
spy = spy.set_index("date").sort_index()
month_end = spy["adj_close"].resample("ME").last()
month_end.head(3)

**`pct_change` does the shifting for you**

In [ ]:
prices = stocks.pivot(index="date", columns="ticker", values="adj_close")
rets = prices.pct_change(fill_method=None)
rets.iloc[:3, :4]

**Growth of £1**

In [ ]:
growth = (1 + rets.fillna(0)).cumprod()
growth.iloc[-1].sort_values(ascending=False)

**Ground truth — check yourself**

In [ ]:
p = prices.loc["2010":"2019"]
(p.iloc[-1] / p.iloc[0]).sort_values(ascending=False)

**Your first matplotlib chart**

In [ ]:
ax = growth.plot(logy=True, figsize=(10, 4.5),
                 xlabel="", ylabel="Value of £1 (log scale)",
                 title="Growth of £1 invested, January 2010 to snapshot date")

**Reading the call — matplotlib's two registers**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5))
for ticker in growth.columns:
    ax.plot(growth.index, growth[ticker], label=ticker)
ax.set_yscale("log")     # a 10% move is the same size in 2010 and 2025
ax.set_ylabel("Value of £1 (log scale)")
ax.set_title("Growth of £1 invested, January 2010 to snapshot date")
ax.legend()
plt.show()

**Step 2 — GENERATE: the assistant's answer**

In [ ]:
def max_drawdown(prices):
    """Largest peak-to-trough decline, as a negative fraction."""
    peak = prices.iloc[0]
    worst = 0.0
    for price in prices.iloc[1:-1]:   # skip first (initial peak) and last
        peak = max(peak, price)
        worst = min(worst, price / peak - 1)
    return worst

**Step 3 — ASSERT: tests we wrote before looking**

In [ ]:
rising = pd.Series([100.0, 105.0, 112.0])         # never falls -> 0.0
crash_at_end = pd.Series([100.0, 110.0, 55.0])    # 110 -> 55 = -0.5

**Step 3 — ASSERT: tests we wrote before looking**

In [ ]:
assert max_drawdown(rising) == 0.0, "rising market should give 0.0"
assert max_drawdown(crash_at_end) == -0.5, "must catch a crash on the final day"

**The fix — and it is simpler, too**

In [ ]:
def max_drawdown(prices):
    """Largest peak-to-trough decline, as a negative fraction."""
    drawdown = prices / prices.cummax() - 1
    return float(drawdown.min())

assert max_drawdown(rising) == 0.0
assert max_drawdown(crash_at_end) == -0.5
print("both assertions pass")

**Now on real data**

In [ ]:
spy_prices = (stocks.loc[stocks["ticker"] == "SPY"]
              .set_index("date").sort_index()["adj_close"])
dd = spy_prices / spy_prices.cummax() - 1
max_drawdown(spy_prices), dd.idxmin()

# W03 — Data Quality, Identifiers and SQL

### W03 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**The dataset for the rest of the unit**

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")
crsp.head(3)

**The dataset for the rest of the unit**

In [ ]:
print(f"{len(crsp):,} stock-months, {crsp['permno'].nunique():,} firms, "
      f"{crsp['date'].min():%Y-%m} to {crsp['date'].max():%Y-%m}")

**Problem 1 — missing months**

In [ ]:
obs = crsp.groupby("permno")["date"].agg(["count", "min", "max"])
months_spanned = (
    obs["max"].dt.to_period("M").astype("int64")
    - obs["min"].dt.to_period("M").astype("int64") + 1
)
gap_share = (months_spanned > obs["count"]).mean()
print(f"Share of firms with at least one internal gap: {gap_share:.1%}")

**Problem 2 — negative prices?**

In [ ]:
raw = pd.DataFrame({
    "permno": [10001] * 3,
    "date": pd.to_datetime(["2019-01-31", "2019-02-28", "2019-03-29"]),
    "prc": [7.1200, -7.0500, 7.2000],   # illustrative excerpt of the raw convention
})
raw

**Problem 3 — corporate actions: splits**

In [ ]:
aapl = crsp[(crsp["permno"] == 14593)
            & crsp["date"].between("2020-06-30", "2020-10-31")]
aapl[["date", "prc", "shrout", "ret"]]

**Adjusted for returns, unadjusted for size**

In [ ]:
snap = crsp[crsp["date"] == "2021-12-31"]
snap.nlargest(5, "mktcap")[["permno", "prc", "shrout", "mktcap"]]

**Problem 4 — outliers and winsorisation**

In [ ]:
xs = crsp.loc[crsp["date"] == "2020-04-30", "ret"].dropna()
lo, hi = xs.quantile([0.01, 0.99])
comparison = pd.DataFrame({"raw": xs.describe(), "winsorised": xs.clip(lo, hi).describe()})
comparison.loc[["count", "mean", "std", "min", "max"]]

**Survivorship: the firms you cannot see**

In [ ]:
ever = crsp["permno"].nunique()
alive_at_end = crsp.loc[crsp["date"] == crsp["date"].max(), "permno"].nunique()
print(f"Firms ever in the panel:    {ever:>6,}")
print(f"Alive at the end (2022-03): {alive_at_end:>6,}")
print(f"Share no longer listed:     {1 - alive_at_end / ever:.0%}")

**Delisting bias: the last return is missing**

In [ ]:
delist_path = DATA / "crsp_delist.parquet"
if delist_path.exists():
    dl = pd.read_parquet(delist_path)
    perf = dl[dl["dlstcd"].between(400, 599)]  # liquidations & performance drops
    print(f"Performance-related delistings: {len(perf):,}")
    print(f"Mean delisting return: {perf['dlret'].mean():.1%}")
else:
    print("Delisting snapshot not present — published magnitudes:")
    print("  Shumway (1997, JF): mean delisting return about -30% (NYSE/AMEX)")
    print("  Shumway & Warther (1999, JF): about -55% on Nasdaq")

**Why SQL, why now**

In [ ]:
import duckdb
duckdb.sql(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT permno) AS firms
    FROM '{DATA}/crsp_monthly.parquet'
""").df()

**SELECT and WHERE: the grammar**

In [ ]:
duckdb.sql(f"""
    SELECT permno, date, ret, mktcap
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date = DATE '2021-12-31' AND mktcap > 1e9
    ORDER BY mktcap DESC
    LIMIT 5
""").df()

**GROUP BY: settling the poll**

In [ ]:
duckdb.sql(f"""
    SELECT date_part('year', date) AS year,
           COUNT(DISTINCT permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date_part('year', date) IN (1997, 2021)
    GROUP BY year
    ORDER BY year
""").df()

**Stocks per exchange per decade**

In [ ]:
by_decade = duckdb.sql(f"""
    SELECT (date_part('year', date) // 10) * 10 AS decade,
           CASE exchcd WHEN 1 THEN 'NYSE'
                       WHEN 2 THEN 'AMEX'
                       WHEN 3 THEN 'Nasdaq' END AS exchange,
           COUNT(DISTINCT permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    GROUP BY decade, exchange
    ORDER BY decade, exchange
""").df()
by_decade.pivot(index="decade", columns="exchange", values="n_stocks")

**JOIN: adding what the table lacks**

In [ ]:
sic_lookup = pd.DataFrame({
    "sic_lo": [100, 1000, 1500, 2000, 4000, 5000, 5200, 6000, 7000, 9100],
    "sic_hi": [999, 1499, 1799, 3999, 4999, 5199, 5999, 6799, 8999, 9999],
    "division": ["Agriculture", "Mining", "Construction", "Manufacturing",
                 "Transport & Utilities", "Wholesale", "Retail",
                 "Finance & Real Estate", "Services", "Public Administration"],
})
duckdb.sql(f"""
    SELECT l.division, COUNT(DISTINCT c.permno) AS n_firms
    FROM '{DATA}/crsp_monthly.parquet' c
    JOIN sic_lookup l ON c.siccd BETWEEN l.sic_lo AND l.sic_hi
    WHERE date_part('year', c.date) = 2021
    GROUP BY l.division
    ORDER BY n_firms DESC
""").df()

**The reveal**

In [ ]:
duckdb.sql(f"""
    SELECT COUNT(permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date_part('year', date) = 1997
""").df()

# W04 — Portfolio Construction and the 1/N Puzzle

### W04 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**The data: CRSP monthly**

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")
crsp = crsp.sort_values(["permno", "date"])

print(crsp.shape)
crsp[["permno", "date", "ret", "prc", "shrout", "mktcap"]].head(3)

**Equal weighting: one line**

In [ ]:
ew = crsp.groupby("date")["ret"].mean()
ew.tail(3)

**Value weighting in pandas**

In [ ]:
# Lag WITHIN each stock — a plain shift(1) would leak across permnos
crsp["mktcap_lag"] = crsp.groupby("permno")["mktcap"].shift(1)

ok = crsp.dropna(subset=["ret", "mktcap_lag"]).copy()
ok["w_ret"] = ok["mktcap_lag"] * ok["ret"]

grp = ok.groupby("date")
vw = grp["w_ret"].sum() / grp["mktcap_lag"].sum()
vw.tail(3)

**Did we get it right?**

In [ ]:
mkt = pd.read_parquet(DATA / "market_monthly.parquet")
mkt = mkt.set_index("date")

compare = pd.DataFrame({"our_vw": vw, "vwretd": mkt["vwretd"],
                        "our_ew": ew, "ewretd": mkt["ewretd"]}).dropna()
print(f"max |our VW - vwretd| = {compare['our_vw'].sub(compare['vwretd']).abs().max():.2e}")
print(f"max |our EW - ewretd| = {compare['our_ew'].sub(compare['ewretd']).abs().max():.2e}")

**Growth of £1: EW vs VW**

In [ ]:
import matplotlib.pyplot as plt

both = pd.DataFrame({"Equal-weighted": ew, "Value-weighted": vw}).dropna()
growth = (1 + both).cumprod()

ax = growth.plot(logy=True, color=["#0F6BBA", "#0F766E"],
                 figsize=(10, 4.5))
ax.set_xlabel("")
ax.set_ylabel("Value of £1 (log scale)")
ax.set_title("All US common stocks, 1990–2022")
plt.show()

**Five stocks, monthly returns**

In [ ]:
px = pd.read_parquet(DATA / "stocks_daily.parquet")

tickers = ["AAPL", "MSFT", "GOOG", "NVDA", "JPM"]
prices = (px[px["ticker"].isin(tickers)]
          .pivot(index="date", columns="ticker", values="adj_close"))

monthly = prices.resample("ME").last().pct_change().dropna()
monthly.tail(3)

**MVO in eight lines**

In [ ]:
def mvo_weights(rets, gamma=10.0):
    mu = rets.mean().to_numpy()
    Sigma = rets.cov().to_numpy()
    ones = np.ones(len(mu))
    Sinv_mu = np.linalg.solve(Sigma, mu)    # solve, don't invert: cheaper, stabler
    Sinv_1 = np.linalg.solve(Sigma, ones)
    lam = (ones @ Sinv_mu - gamma) / (ones @ Sinv_1)
    w = (Sinv_mu - lam * Sinv_1) / gamma
    assert np.isclose(w.sum(), 1.0)         # Week 02 habit: assert before you trust
    return w

w = mvo_weights(monthly)
pd.Series(w, index=monthly.columns, name="weight")

**The efficient frontier**

In [ ]:
records = []
for gamma in np.arange(2, 61, 2):
    wg = mvo_weights(monthly, gamma)
    records.append({"vol": float(np.sqrt(wg @ monthly.cov() @ wg)),
                    "ret": float(wg @ monthly.mean())})
frontier = pd.DataFrame(records)

w_1n = np.full(len(tickers), 1 / len(tickers))
naive = {"vol": float(np.sqrt(w_1n @ monthly.cov() @ w_1n)),
         "ret": float(w_1n @ monthly.mean())}

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(frontier["vol"], frontier["ret"], color="#0F6BBA", marker="o")
ax.scatter(naive["vol"], naive["ret"], color="#0F766E", s=60, zorder=3)
ax.annotate("1/N", (naive["vol"], naive["ret"]), color="#0F766E",
            xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xlabel("Monthly volatility")
ax.set_ylabel("Monthly expected return")
ax.set_title("Efficient frontier (in-sample), with 1/N inside it")
plt.show()

**Scaling up: 600 stocks**

In [ ]:
window = crsp[(crsp["date"] >= "2012-01-01") & (crsp["date"] <= "2021-12-31")]
counts = window.groupby("permno")["ret"].count()
full_history = counts[counts == 120].index

big = (window[window["permno"].isin(full_history)]
       .pivot(index="date", columns="permno", values="ret")
       .iloc[:, :600])
print(big.shape)

**The designed failure**

In [ ]:
Sigma_big = big.cov().to_numpy()
print(f"condition number of Sigma: {np.linalg.cond(Sigma_big):.2e}")

**The designed failure**

In [ ]:
try:
    w_big = mvo_weights(big)
    print(f"largest |weight|: {np.abs(w_big).max():,.0f}")
except np.linalg.LinAlgError as e:
    print(f"LinAlgError: {e}")
except AssertionError:
    print("AssertionError: weights do not sum to one — solve() returned garbage")

**The backtest loop**

In [ ]:
WINDOW, GAMMA, COST = 60, 10.0, 0.0010
n = monthly.shape[1]
results, prev = [], {"1/N": None, "MVO": None}

for t in range(WINDOW, len(monthly)):
    train = monthly.iloc[t - WINDOW:t]         # strictly before month t: no peeking
    weights = {"1/N": np.full(n, 1 / n), "MVO": mvo_weights(train, GAMMA)}
    for name, wt in weights.items():
        gross = float(wt @ monthly.iloc[t])
        to = 1.0 if prev[name] is None else float(np.abs(wt - prev[name]).sum())
        results.append({"date": monthly.index[t], "strategy": name,
                        "gross": gross, "turnover": to,
                        "net": gross - COST * to})
        prev[name] = wt

bt = pd.DataFrame(results)

**The reveal: growth of £1, net of costs**

In [ ]:
# one column per strategy, then compound down each column
net_bt = bt.pivot(index="date", columns="strategy", values="net")
growth_bt = (1 + net_bt).cumprod()

ax = growth_bt.plot(color=["#0F6BBA", "#0F766E"], figsize=(10, 4.5))
ax.set_xlabel("")
ax.set_ylabel("Value of £1 (net of 10bps costs)")
ax.set_title("Out of sample, the optimiser meets reality")
plt.show()

**The scoreboard**

In [ ]:
def ann_sharpe(x):
    return np.sqrt(12) * x.mean() / x.std()

scoreboard = bt.groupby("strategy").agg(
    sharpe_gross=("gross", ann_sharpe),
    sharpe_net=("net", ann_sharpe),
    avg_turnover=("turnover", "mean"))
scoreboard

**AI checkpoint: the reveal**

In [ ]:
ai = crsp.dropna(subset=["ret", "mktcap"]).copy()
ai["w_ret"] = ai["mktcap"] * ai["ret"]
vw_ai = ai.groupby("date")["w_ret"].sum() / ai.groupby("date")["mktcap"].sum()

common = vw.index.intersection(vw_ai.index)
print(f"honest VW,     annualised mean: {12 * vw.loc[common].mean():.2%}")
print(f"look-ahead VW, annualised mean: {12 * vw_ai.loc[common].mean():.2%}")

# W05 — Asset Pricing: CAPM, Factors, and How to Test Them

### W05 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**This week's lab bench**

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")   # US common stocks, NYSE/AMEX/NASDAQ
mkt  = pd.read_parquet(DATA / "market_monthly.parquet") # the VW index you built in W04
ff   = pd.read_parquet(DATA / "ff_factors_monthly.parquet")  # Ken French factors + rf, decimal units

print(crsp["date"].min().date(), "to", crsp["date"].max().date(),
      "|", crsp["permno"].nunique(), "permnos")

**Step 1: excess returns**

In [ ]:
rf = ff[["date", "rf"]]

panel = crsp.merge(rf, on="date", how="inner", validate="many_to_one")
panel["exret"] = panel["ret"] - panel["rf"]

mkt = mkt.merge(rf, on="date", how="inner", validate="one_to_one")
mkt["exmkt"] = mkt["vwretd"] - mkt["rf"]

panel = panel.merge(mkt[["date", "exmkt"]], on="date", validate="many_to_one")
panel = panel.dropna(subset=["exret", "exmkt"])

print(f"Mean market excess return: {mkt['exmkt'].mean():.4f} per month")

**Step 2: one beta per stock**

In [ ]:
# Require at least 36 months so the regression means something
counts = panel.groupby("permno")["exret"].size()
keep = counts[counts >= 36].index
sub = panel[panel["permno"].isin(keep)]

def est_beta(g):
    return np.polyfit(g["exmkt"], g["exret"], 1)[0]  # slope of the OLS line

betas = sub.groupby("permno")[["exret", "exmkt"]].apply(est_beta)
mean_exret = sub.groupby("permno")["exret"].mean()

cs = pd.DataFrame({"beta": betas, "mean_exret": mean_exret}).dropna()
print(f"{len(cs)} stocks with estimated betas")

**Step 3: sanity-check before you trust**

In [ ]:
print(cs["beta"].describe())

**Step 4: the cross-sectional test**

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(cs["beta"])
sml = sm.OLS(cs["mean_exret"], X).fit()

print(f"Intercept: {sml.params['const']:>8.4f}   (t = {sml.tvalues['const']:.2f})")
print(f"Slope:     {sml.params['beta']:>8.4f}   (t = {sml.tvalues['beta']:.2f})")
print(f"CAPM says the slope should be about {mkt['exmkt'].mean():.4f}")

**The verdict, in one picture**

In [ ]:
import matplotlib.pyplot as plt

cs["beta_dec"] = pd.qcut(cs["beta"], 10, labels=False, duplicates="drop") + 1
dec = cs.groupby("beta_dec")[["beta", "mean_exret"]].mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axline((0, 0), slope=mkt["exmkt"].mean(),
          color="#0F766E", linestyle="--")              # CAPM prediction
ax.axline((0, sml.params["const"]), slope=sml.params["beta"],
          color="#0F6BBA")                              # the data's fitted line
ax.scatter(dec["beta"], dec["mean_exret"], color="#0F6BBA", s=45, zorder=3)
ax.set_xlabel("Estimated beta (decile average)")
ax.set_ylabel("Mean monthly excess return")
ax.set_title("The SML on trial: dashed teal = CAPM prediction, solid blue = the data")
plt.show()

**Fama–MacBeth in six lines**

In [ ]:
from linearmodels import FamaMacBeth

fm_panel = sub.merge(cs[["beta"]], left_on="permno", right_index=True)
fm_panel = fm_panel.set_index(["permno", "date"])          # entity, then time
fm_res = FamaMacBeth.from_formula("exret ~ 1 + beta", data=fm_panel).fit(
    cov_type="kernel")                                      # Newey–West on the lambdas
print(fm_res.params); print(fm_res.tstats)

**Verify, don't trust**

In [ ]:
crsp = crsp.sort_values(["permno", "date"])
crsp["mktcap_lag"] = crsp.groupby("permno")["mktcap"].shift(1)  # lagged: no look-ahead

sizes = crsp.dropna(subset=["mktcap_lag", "ret"]).copy()
sizes["size_dec"] = (sizes.groupby("date")["mktcap_lag"]
                     .transform(lambda x: pd.qcut(x, 10, labels=False,
                                                  duplicates="drop"))
                     .astype(int) + 1)

size_pattern = (sizes.groupby(["date", "size_dec"])["ret"].mean()  # EW within decile
                .groupby("size_dec").mean() * 100)
print(size_pattern.round(2))

**The factors are public goods**

In [ ]:
print(ff.tail(3))
ff_sample = ff.set_index("date").loc["1990-01":"2022-03", ["mkt_rf", "smb", "hml", "mom"]]
print((ff_sample.mean() * 100).round(2), "  <- mean % per month, Jan 1990 to Mar 2022")

**Factors as data reduction**

In [ ]:
import statsmodels.formula.api as smf

ff25 = pd.read_parquet(DATA / "ff25_monthly.parquet")
ff25 = ff25.merge(ff, on="date", validate="many_to_one")
ff25["exret"] = ff25["ret"] - ff25["rf"]

def fit_r2(g):
    return pd.Series({
        "r2_capm": smf.ols("exret ~ mkt_rf", data=g).fit().rsquared,
        "r2_ff3":  smf.ols("exret ~ mkt_rf + smb + hml", data=g).fit().rsquared})

# the same split-apply-combine as the betas, now two regressions per portfolio
r2 = ff25.groupby(["size_q", "bm_q"])[["exret", "mkt_rf", "smb", "hml"]].apply(fit_r2)
print(r2[["r2_capm", "r2_ff3"]].mean().round(2))

**$R^2 \rightarrow 1$**

In [ ]:
print("CAPM R² by portfolio (rows = size quintile, cols = B/M quintile):")
print(r2["r2_capm"].unstack("bm_q").round(2))   # unstack: bm_q moves from the index into the columns
print("\nFF3 R²:")
print(r2["r2_ff3"].unstack("bm_q").round(2))

**AI checkpoint: mark the machine**

In [ ]:
sv = ff25[(ff25["size_q"] == 1) & (ff25["bm_q"] == 5)]
fit = smf.ols("exret ~ mkt_rf + smb + hml", data=sv).fit()
print(fit.summary().tables[1])
print(f"R-squared: {fit.rsquared:.3f}")

# W07 — Performance Evaluation & Backtest Pathology

### W07 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**The contenders, on one chart**

In [ ]:
import matplotlib.pyplot as plt

mkt = pd.read_parquet(DATA / "market_monthly.parquet")
ff  = pd.read_parquet(DATA / "ff_factors_monthly.parquet")
df = mkt.merge(ff, on="date", how="inner", validate="one_to_one").set_index("date")

wealth = pd.DataFrame({
    "EW (1/N)":    (1 + df["ewretd"]).cumprod(),
    "VW (market)": (1 + df["vwretd"]).cumprod(),
})

wealth.plot(
    logy=True, color=["#0F6BBA", "#0F766E"], figsize=(8, 4),
    title="EW vs VW, CRSP common stocks, 1990–2022",
    ylabel="Growth of $1 (log scale)", xlabel="")
plt.show()

**Alpha in practice: CAPM, then FF3**

In [ ]:
import statsmodels.formula.api as smf

df["strat_ex"] = df["ewretd"] - df["rf"]   # strategy: 1/N portfolio
capm = smf.ols("strat_ex ~ mkt_rf", data=df).fit(
    cov_type="HAC", cov_kwds={"maxlags": 6})
ff3 = smf.ols("strat_ex ~ mkt_rf + smb + hml", data=df).fit(
    cov_type="HAC", cov_kwds={"maxlags": 6})

print(f"CAPM alpha: {12 * capm.params['Intercept']:.2%} p.a. "
      f"(t = {capm.tvalues['Intercept']:.2f})")
print(f"FF3  alpha: {12 * ff3.params['Intercept']:.2%} p.a. "
      f"(t = {ff3.tvalues['Intercept']:.2f})")
print(ff3.summary().tables[1])

**Computing the ratios**

In [ ]:
AF = 12                                      # monthly data
ex        = df["ewretd"] - df["rf"]          # strategy excess return
bench_ex  = df["vwretd"] - df["rf"]          # benchmark excess return
active    = df["ewretd"] - df["vwretd"]      # active return vs benchmark

beta = np.cov(ex, bench_ex, ddof=1)[0, 1] / bench_ex.var()
ratios = pd.Series({
    "Sharpe (ann.)":      np.sqrt(AF) * ex.mean() / ex.std(),
    "Sortino (ann.)":     np.sqrt(AF) * ex.mean() / ex[ex < 0].std(),
    "Information (ann.)": np.sqrt(AF) * active.mean() / active.std(),
    "Treynor (ann.)":     AF * ex.mean() / beta,
    "Benchmark Sharpe":   np.sqrt(AF) * bench_ex.mean() / bench_ex.std(),
})
print(ratios.round(3))

**Drawdown in six lines**

In [ ]:
import matplotlib.pyplot as plt

W   = (1 + df["ewretd"]).cumprod()
hwm = W.cummax()
dd  = W / hwm - 1

fig, ax = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
ax[0].plot(W.index, W, color="#0F6BBA", label="Wealth")
ax[0].plot(hwm.index, hwm, color="#0F766E", ls="--", label="High-water mark")
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_ylabel("Growth of $1")
ax[1].fill_between(dd.index, dd, 0, color="#B91C1C", alpha=0.6)
ax[1].set_ylabel("Drawdown")
print(f"Maximum drawdown: {dd.min():.1%}, dated {dd.idxmin():%Y-%m}")
plt.tight_layout()
plt.show()

**Poll reveal: Sharpe ratios have standard errors**

In [ ]:
def sharpe_tstat(sr_annual, years, periods=12):
    sr_m = sr_annual / np.sqrt(periods)          # de-annualise
    T = years * periods
    se_m = np.sqrt((1 + 0.5 * sr_m**2) / T)      # Lo (2002)
    return sr_m / se_m

print(f"Fund A (SR 1.1, 3y):  t = {sharpe_tstat(1.1, 3):.2f}")
print(f"Fund B (SR 0.7, 15y): t = {sharpe_tstat(0.7, 15):.2f}")

**Lie 3 → Lie 4: short samples become period dependence**

In [ ]:
df["decade"] = (df.index.year // 10) * 10
g = (df["vwretd"] - df["rf"]).groupby(df["decade"])
by_decade = pd.DataFrame({
    "ann_sharpe": np.sqrt(12) * g.mean() / g.std(),
    "months": g.count(),
})
print(by_decade.round(2))

**Gross numbers are fiction (W04 reprise)**

In [ ]:
costs = pd.DataFrame({
    "strategy": ["Momentum tilt", "1/N (monthly rebal.)", "VW market"],
    "gross_pa": [0.1020, 0.0980, 0.0910],
    "turnover_pa": [5.00, 0.20, 0.05],   # two-way (sum of |Δw|), as multiple of portfolio value
})
costs["cost_drag"] = costs["turnover_pa"] * 0.0010
costs["net_pa"] = costs["gross_pa"] - costs["cost_drag"]
print(costs.sort_values("net_pa", ascending=False).to_string(index=False))

**Brinson–Fachler, computed**

In [ ]:
bf = pd.DataFrame({
    "sector":  ["Banking", "Technology"],
    "w_port":  [0.60, 0.40], "r_port":  [0.15, 0.20],
    "w_bench": [0.70, 0.30], "r_bench": [0.12, 0.08],
})
R = (bf["w_port"] * bf["r_port"]).sum()      # 17.0%
B = (bf["w_bench"] * bf["r_bench"]).sum()    # 10.8%

bf["allocation"]  = (bf["w_port"] - bf["w_bench"]) * (bf["r_bench"] - B)
bf["selection"]   = bf["w_bench"] * (bf["r_port"] - bf["r_bench"])
bf["interaction"] = (bf["w_port"] - bf["w_bench"]) * (bf["r_port"] - bf["r_bench"])
print(bf[["sector", "allocation", "selection", "interaction"]].to_string(index=False))
print(f"\nTotals: allocation {bf['allocation'].sum():+.2%}, "
      f"selection {bf['selection'].sum():+.2%}, "
      f"interaction {bf['interaction'].sum():+.2%}  "
      f"→ active return {R - B:+.2%}")

# W08 — Fixed Income & Macro Signals

### W08 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**Forward rates: what the curve implies**

In [ ]:
r4, r5 = 0.046, 0.048                      # 4y and 5y zero rates
f45 = (1 + r5) ** 5 / (1 + r4) ** 4 - 1    # implied 1y rate, 4y forward
print(f"Implied forward rate f(4,5): {f45:.4%}")

**Loading the curve from our snapshot**

In [ ]:
fred = pd.read_parquet(DATA / "fred_monthly.parquet")

yields = fred.pivot(index="date", columns="series_id", values="value")
print(yields.columns.tolist())
print(yields.index.min().date(), "->", yields.index.max().date())

**Three snapshots of the curve**

In [ ]:
import matplotlib.pyplot as plt

mat_map = {"DGS3MO": 0.25, "DGS1": 1, "DGS2": 2,
           "DGS5": 5, "DGS10": 10, "DGS30": 30}
curve_data = yields[list(mat_map)].dropna()
last_date = curve_data.index.max()

dates = [pd.Timestamp("2021-06-30"), pd.Timestamp("2023-06-30"), last_date]
curves = curve_data.loc[dates].T                   # one column per date
curves.index = curves.index.map(mat_map)           # series_id -> maturity in years
curves.columns = curves.columns.strftime("%b %Y")

ax = curves.plot(marker="o", figsize=(8, 4.2), xticks=list(mat_map.values()))
ax.set_title("CMT yield curves from the unit snapshot")
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Yield (% p.a.)")
ax.legend(title="Date")
plt.show()

**The term spread and inversion**

In [ ]:
import matplotlib.pyplot as plt

spread_m = yields["T10Y3M"].dropna()

ax = spread_m.plot(color="#0F6BBA", linewidth=0.7, figsize=(8, 4))
ax.axhline(0, color="#d62728", linestyle="--")
ax.set_title("10-year minus 3-month Treasury spread")
ax.set_xlabel("")
ax.set_ylabel("Spread (percentage points)")
plt.show()

**FETCH: the quarterly data**

In [ ]:
fredq = pd.read_parquet(DATA / "fred_quarterly.parquet")

q = fredq.pivot(index="date", columns="series_id", values="value")
q.index = q.index.to_period("Q")   # calendar quarter = the join key
print(q.tail(3))

**TRANSFORM I: future growth at horizon $k$**

In [ ]:
y = q["GDPC1"]
horizons = [1, 2, 3, 4, 6, 8]
growth = pd.DataFrame(
    {f"k{k}": 400 * np.log(y.shift(-k) / y) / k for k in horizons}
)
print(growth.dropna().tail(3))

**TRANSFORM II: the quarterly spread + join gate**

In [ ]:
ts_q = spread_m.groupby(spread_m.index.to_period("Q")).last()
ts_q.name = "spread"                       # quarter-end value, in pp

joint = growth.join(ts_q, how="inner")
print(f"n = {len(joint)}, {joint.index.min()} -> {joint.index.max()}")

**ESTIMATE I: why overlapping observations need HAC**

In [ ]:
import statsmodels.api as sm

rows = []
for k in horizons:
    df = joint[[f"k{k}", "spread"]].dropna()
    res = sm.OLS(df[f"k{k}"], sm.add_constant(df["spread"])).fit(
        cov_type="HAC", cov_kwds={"maxlags": max(k, 4)})
    rows.append({"horizon_q": k, "slope": res.params["spread"],
                 "t_hac": res.tvalues["spread"], "n": int(res.nobs)})
print(pd.DataFrame(rows).to_string(index=False))

**ESTIMATE II: the recession probit**

In [ ]:
probit_df = q[["USRECQ"]].join(ts_q.shift(4), how="inner").dropna()
X = sm.add_constant(probit_df["spread"])
probit_res = sm.Probit(probit_df["USRECQ"], X).fit(disp=0)
print(probit_res.summary().tables[1])

**Coefficients are not probabilities**

In [ ]:
print(probit_res.get_margeff().summary_frame())

**The money chart: fitted probabilities**

In [ ]:
import matplotlib.pyplot as plt

plot_df = probit_df.assign(prob=probit_res.predict(X))
plot_df.index = plot_df.index.to_timestamp()     # quarter -> its first day, for the date axis

fig, ax = plt.subplots(figsize=(8, 4.2))
for start in plot_df.loc[plot_df["USRECQ"] == 1].index:
    ax.axvspan(start, start + pd.DateOffset(months=3),
               color="#d62728", alpha=0.25, linewidth=0)
ax.plot(plot_df.index, plot_df["prob"], color="#0F6BBA", linewidth=1)
fig.suptitle("Recession probability from the 4-quarter-lagged term spread")
ax.set_title("Probit fitted values; NBER recessions shaded", fontsize=10)
ax.set_ylabel("Fitted probability")
plt.show()

**Poll reveal**

In [ ]:
print(f"Latest quarter in snapshot: {plot_df.index[-1].date()}")
print(f"Spread 4 quarters earlier:  {probit_df['spread'].iloc[-1]:.2f} pp")
print(f"Fitted recession probability: {plot_df['prob'].iloc[-1]:.1%}")

# W09 — Text as Financial Data I

### W09 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**Meet the data**

In [ ]:
sents = pd.read_parquet(DATA / "edgar_10k_sentences.parquet")
print(sents.shape)
print(sents["company"].unique())
sents[["company", "filing_year", "item", "sent_id"]].head(3)

**Meet the data**

In [ ]:
# What does a risk-factor sentence look like?
print(sents["text"].iloc[0][:300])

**Why generic dictionaries fail: the "liability" demo**

In [ ]:
lm = pd.read_parquet(DATA / "lm_dictionary.parquet")
lm["word"] = lm["word"].str.lower()
neg_words = set(lm.loc[lm["sentiment"] == "negative", "word"])
pos_words = set(lm.loc[lm["sentiment"] == "positive", "word"])

generic_flags = ["liability", "tax", "cost", "capital", "board", "foreign"]
finance_negs  = ["loss", "impairment", "litigation", "default", "adverse"]
for w in generic_flags + finance_negs:
    print(f"{w:<12} in LM negative list: {w in neg_words}")

**The LM dictionary at a glance**

In [ ]:
lm["sentiment"].value_counts()

**pandas string wrangling: from text to tokens**

In [ ]:
# Tokenise: one row per (sentence, word)
words = (
    sents.assign(word=sents["text"].str.lower().str.findall(r"[a-z]+"))
         .explode("word")
         .dropna(subset=["word"])
)
words["neg"] = words["word"].isin(neg_words)
words["pos"] = words["word"].isin(pos_words)
print(f"{len(words):,} tokens from {len(sents):,} sentences")
words.loc[words["neg"], "word"].value_counts().head(5)

**Firm-level sentiment: who sounds most worried?**

In [ ]:
tone = (
    words.groupby(["company", "filing_year"], as_index=False)
         .agg(n_words=("word", "size"),
              n_neg=("neg", "sum"),
              n_pos=("pos", "sum"))
)
tone["neg_per_1000"] = 1000 * tone["n_neg"] / tone["n_words"]
latest_year = tone["filing_year"].mode().max()   # the year most firms filed in
latest = tone.loc[tone["filing_year"] == latest_year]
latest.sort_values("neg_per_1000", ascending=False).head(10)

**The picture**

In [ ]:
import matplotlib.pyplot as plt

ax = (latest.set_index("company")["neg_per_1000"]
            .sort_values()
            .plot.barh(color="#0F6BBA", figsize=(9, 4.5)))
ax.set_xlabel("LM-negative words per 1,000 words")
ax.set_ylabel("")
ax.set_title(f"Risk-factor tone, {latest_year} filings\n"
             "Item 1A, Loughran–McDonald negative list")
plt.tight_layout()
plt.show()

**Searching for themes: `str.contains`**

In [ ]:
themes = {
    "cyber":        r"cyber",
    "climate":      r"climate|environmental",
    "supply chain": r"supply chain|supplier",
    "pandemic":     r"pandemic|epidemic|covid",
}
by_theme = {}
for name, pattern in themes.items():
    hits = sents["text"].str.contains(pattern, case=False, regex=True)
    by_year = sents.loc[hits].groupby("filing_year").size()
    total = sents.groupby("filing_year").size()
    share = (1000 * by_year / total).round(1)
    by_theme[name] = share
pd.DataFrame(by_theme).fillna(0).T

**Validate the cached output**

In [ ]:
import json
# Cached output from the call above (run once by the convenor, July 2026),
# so this runs offline and gives everyone the same result.
cached = ('{"risk_type": "operational", "sentiment": "negative", '
          '"mentions_quantity": false}')
extracted = json.loads(cached)

ALLOWED_RISK = {"operational", "financial", "legal",
                "macro", "technology", "other"}
ALLOWED_SENT = {"positive", "negative", "neutral"}

assert set(extracted) == {"risk_type", "sentiment", "mentions_quantity"}
assert extracted["risk_type"] in ALLOWED_RISK
assert extracted["sentiment"] in ALLOWED_SENT
assert isinstance(extracted["mentions_quantity"], bool)
extracted

**What does it cost? Token accounting**

In [ ]:
tokens_per_word = 4 / 3          # rule of thumb: 1 word ≈ 1.33 tokens
item1a_words   = 12_000          # a typical Item 1A section
input_tokens   = item1a_words * tokens_per_word
output_tokens  = 300             # our JSON schema is tiny
price_in, price_out = 3.00, 15.00   # USD per million tokens (Sonnet-class, 2026)

cost_per_filing = (input_tokens * price_in
                   + output_tokens * price_out) / 1e6
n_filings = 3_500                # ballpark: one year of US-listed 10-Ks
print(f"One filing:                ${cost_per_filing:.3f}")
print(f"One year of the universe:  ${cost_per_filing * n_filings:,.0f}")
print(f"Thirty years:              ${cost_per_filing * n_filings * 30:,.0f}")

# W10 — Text as Financial Data II: Measurement and Validation

### W10 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

**Kappa: a worked example**

In [ ]:
from sklearn.metrics import cohen_kappa_score

rita = (["neg"] * 6) + (["neu"] * 8) + (["pos"] * 6)
sam  = ["neg", "neg", "neg", "neg", "neg", "neu",
        "neu", "neu", "neu", "neu", "neu", "neu", "neg", "pos",
        "pos", "pos", "pos", "pos", "neu", "neu"]

agreement = np.mean([a == b for a, b in zip(rita, sam)])
kappa = cohen_kappa_score(rita, sam)
print(f"raw agreement: {agreement:.2f}   Cohen's kappa: {kappa:.2f}")

**The evaluation data**

In [ ]:
gold = pd.read_parquet(DATA / "labeled_sentences.parquet")
print(gold.shape)
print(gold["label_sentiment"].value_counts())
gold[["sent_id", "text", "label_sentiment"]].head(3)

**The baseline: rebuild the W09 dictionary classifier**

In [ ]:
import re

lm = pd.read_parquet(DATA / "lm_dictionary.parquet")
pos_words = set(lm.loc[lm["sentiment"] == "positive", "word"].str.lower())
neg_words = set(lm.loc[lm["sentiment"] == "negative", "word"].str.lower())

def lm_label(text: str) -> str:
    words = re.findall(r"[a-z']+", text.lower())
    p = sum(w in pos_words for w in words)
    n = sum(w in neg_words for w in words)
    if p > n:
        return "positive"
    if n > p:
        return "negative"
    return "neutral"

gold["pred_lm"] = gold["text"].map(lm_label)
gold["pred_lm"].value_counts()

**The confusion matrix**

In [ ]:
from sklearn.metrics import confusion_matrix

LABELS = ["negative", "neutral", "positive"]
cm = confusion_matrix(gold["label_sentiment"], gold["pred_lm"], labels=LABELS)
pd.DataFrame(cm,
             index=[f"true {l}" for l in LABELS],
             columns=[f"pred {l}" for l in LABELS])

**The full report**

In [ ]:
from sklearn.metrics import classification_report, cohen_kappa_score

print(classification_report(gold["label_sentiment"], gold["pred_lm"],
                            labels=LABELS, digits=2, zero_division=0))
kappa_lm = cohen_kappa_score(gold["label_sentiment"], gold["pred_lm"])
print(f"kappa vs gold: {kappa_lm:.2f}")

**Cached outputs: ten sentences, three opinions**

In [ ]:
bake = pd.DataFrame({
    "text": [
        "The company recorded a significant impairment loss in the fourth quarter.",
        "Operating margins improved and the segment returned to profitability.",
        "We do not expect these losses to have a material adverse effect on our results.",
        "The settlement did not result in any impairment of goodwill.",
        "Our insurance business benefits when claims decline.",
        "Changes in interest rates may adversely affect the fair value of our investments.",
        "The timing of regulatory approval remains uncertain.",
        "Revenue was flat compared with the prior year.",
        "Our competitors have announced significant losses in this segment.",
        "The notes are subject to early redemption at the option of the issuer.",
    ],
    "gold": ["negative", "positive", "neutral", "neutral", "positive",
             "negative", "neutral", "neutral", "positive", "neutral"],
    "dict": ["negative", "positive", "negative", "negative", "neutral",
             "negative", "neutral", "neutral", "negative", "neutral"],
    "llm":  ["negative", "positive", "neutral", "neutral", "positive",
             "negative", "negative", "mixed", "negative", "neutral"],
})
bake

**Scoring the bake-off**

In [ ]:
print(f"dictionary accuracy: {(bake['dict'] == bake['gold']).mean():.0%}")
print(f"LLM accuracy:        {(bake['llm']  == bake['gold']).mean():.0%}")

negation = bake.iloc[[2, 3, 4]]   # the negation / context sentences
print(f"\nnegation subset — dictionary: "
      f"{(negation['dict'] == negation['gold']).mean():.0%}, "
      f"LLM: {(negation['llm'] == negation['gold']).mean():.0%}")

**AI checkpoint: recompute**

In [ ]:
y_true = ["negative"] * 10 + ["neutral"] * 180 + ["positive"] * 10
y_pred = (["negative"] * 4 + ["neutral"] * 6          # true negative
          + ["negative"] * 1 + ["neutral"] * 179       # true neutral
          + ["neutral"] * 5 + ["positive"] * 5)        # true positive

print(classification_report(y_true, y_pred, labels=LABELS,
                            digits=2, zero_division=0))

**Step 1: firm-year tone (computed)**

In [ ]:
sents = pd.read_parquet(DATA / "edgar_10k_sentences.parquet")
sents["pred_lm"] = sents["text"].map(lm_label)
sents["is_neg"] = sents["pred_lm"] == "negative"

tone = (sents.groupby(["cik", "company", "filing_year"], as_index=False)
             .agg(n_sentences=("sent_id", "size"),
                  neg_share=("is_neg", "mean")))
tone.sort_values("neg_share", ascending=False).head(8)

**Step 1: firm-year tone (computed)**

In [ ]:
import matplotlib.pyplot as plt

ax = (tone.pivot(index="company", columns="filing_year", values="neg_share")
          .plot.barh(color=["#0F6BBA", "#0F766E", "#9CA3AF", "#D97706"], figsize=(10, 4.5)))
ax.set_xlabel("Share of negative Item 1A sentences")
ax.set_ylabel("")
ax.legend(title="Filing year")
plt.show()

# W11 — Communicating Financial Data: Visualisation, Memos and the Showcase

### W11 setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.options.display.float_format = "{:.4f}".format

**Exhibit A**

In [ ]:
px = pd.read_parquet(DATA / "stocks_daily.parquet")
pair = (px[px["ticker"].isin(["AAPL", "NVDA"])]
          .pivot(index="date", columns="ticker", values="close"))
ax = pair.plot(figsize=(9, 4), title="Which of our stocks performed better?")
ax.set_xlabel(""); ax.set_ylabel("Price ($)")
ax.legend(title=None)
plt.show()

**Exhibit B**

In [ ]:
prices = px.pivot(index="date", columns="ticker", values="adj_close")
ann = prices.pct_change(fill_method=None).mean() * 252
ax = ann.plot.bar(color="#0F6BBA", legend=False, rot=0,
                  figsize=(9, 4), title="Annual returns of our stocks")
ax.set_ylim(0.08, ann.max() * 1.05)
ax.set_xlabel(""); ax.set_ylabel("Annual return")
plt.show()

**The repair: adjusted prices, a £1 start, a log scale — and SPY as the benchmark**

In [ ]:
growth = px.pivot(index="date", columns="ticker", values="adj_close")
growth = growth / growth.iloc[0]
ax = growth.plot(logy=True, figsize=(9, 4),
                 title="Growth of £1, dividends reinvested, 2010 to sample end (log scale)")
ax.set_xlabel(""); ax.set_ylabel("Value of £1 (log scale)")
ax.legend(title=None)
plt.show()